# Ajuste interactivo de zoom/centro de mapas

1. Selecciona mapa en el dropdown
2. Mueve sliders (Zoom, Lat, Lon)
3. Pulsa **Preview** — renderiza a baja res inline
4. Repite para cada mapa
5. Pulsa **Guardar config** — escribe valores a `scripts/export_maps.py`
6. Lanza `python scripts/export_maps.py` para exportar en alta resolución

In [1]:
import sys, re, importlib.util
from pathlib import Path
import ipywidgets as w
from IPython.display import display, clear_output
from html2image import Html2Image

REPO    = Path('..').resolve()
OUTPUTS = REPO / 'outputs'
PREVIEW = OUTPUTS / '_preview'
PREVIEW.mkdir(exist_ok=True)

spec = importlib.util.spec_from_file_location('export_maps', REPO / 'scripts' / 'export_maps.py')
em   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(em)

hti = Html2Image(output_path=str(PREVIEW))
print('Cargados', len(em.MAPS), 'mapas')

Exporting map_heatmap_conf070.html (zoom=orig, center=orig) -> map_heatmap_conf070.png ...
  -> TFG_LateX/figs/map_heatmap_conf070.png
Exporting map_density_conf070.html (zoom=orig, center=orig) -> map_density_conf070.png ...
  -> TFG_LateX/figs/map_density_conf070.png
Exporting map_consistency.html (zoom=13, center=[39.452, -0.379]) -> map_consistency.png ...
  -> TFG_LateX/figs/map_consistency.png
Exporting map_etse_comparison.html (zoom=orig, center=orig) -> map_etse_comparison.png ...
  -> TFG_LateX/figs/map_etse_comparison.png
Exporting map_speed_all.html (zoom=13, center=[39.45, -0.385]) -> map_speed_all.png ...
  -> TFG_LateX/figs/map_speed_all.png
Done.
Cargados 5 mapas


In [2]:
# Estado editable
state = {}
for html_name, png_name, _w2, _h, center, zoom in em.MAPS:
    state[png_name] = dict(
        html = html_name,
        zoom = zoom if zoom else 12.0,
        lat  = center[0] if center else 39.452,
        lon  = center[1] if center else -0.379,
    )

MAP_NAMES = list(state.keys())

# ── Widgets ──────────────────────────────────────────────────────────────────
dd_map  = w.Dropdown(options=MAP_NAMES, description='Mapa:',
                     layout=w.Layout(width='420px'))
sl_zoom = w.FloatSlider(min=11.0, max=13.0, step=0.1, readout_format='.1f',
                        description='Zoom:', style={'description_width': '50px'},
                        layout=w.Layout(width='380px'))
sl_lat  = w.FloatSlider(min=39.20, max=39.70, step=0.001, readout_format='.3f',
                        description='Lat:', style={'description_width': '50px'},
                        layout=w.Layout(width='450px'))
sl_lon  = w.FloatSlider(min=-0.70, max=-0.20, step=0.001, readout_format='.3f',
                        description='Lon:', style={'description_width': '50px'},
                        layout=w.Layout(width='450px'))
btn_prev = w.Button(description='Preview', button_style='primary',
                    layout=w.Layout(width='120px'))
btn_save = w.Button(description='Guardar config', button_style='success',
                    layout=w.Layout(width='160px'))

# Image widget — se actualiza directamente con bytes PNG
img_widget = w.Image(format='png', width=900, height=540)
lbl_status = w.Label(value='Pulsa Preview para renderizar.')


def load_map(change=None):
    s = state[dd_map.value]
    sl_zoom.value = round(s['zoom'], 1)
    sl_lat.value  = round(s['lat'], 3)
    sl_lon.value  = round(s['lon'], 3)


def on_preview(btn=None):
    name = dd_map.value
    s = state[name]
    s['zoom'] = sl_zoom.value
    s['lat']  = sl_lat.value
    s['lon']  = sl_lon.value

    lbl_status.value = f'Renderizando {name}...'

    html_path = OUTPUTS / s['html']
    tmp = em.patch_map(html_path, center=[s['lat'], s['lon']], zoom=s['zoom'])
    out_name = f'preview_{name}'
    hti.screenshot(html_file=str(tmp), save_as=out_name, size=(900, 540))
    tmp.unlink(missing_ok=True)

    img_path = PREVIEW / out_name
    if img_path.exists():
        img_widget.value = img_path.read_bytes()
        lbl_status.value = f'{name} | zoom={s["zoom"]:.1f}  lat={s["lat"]:.3f}  lon={s["lon"]:.3f}'
    else:
        lbl_status.value = 'Error: imagen no generada.'


def on_save(btn=None):
    s = state[dd_map.value]
    s['zoom'] = sl_zoom.value
    s['lat']  = sl_lat.value
    s['lon']  = sl_lon.value

    lines = []
    for html_name, png_name, ww, hh, _c, _z in em.MAPS:
        sv = state[png_name]
        lines.append(
            f'    ("{html_name}", "{png_name}", {ww}, {hh},'
            f' [{sv["lat"]:.3f}, {sv["lon"]:.3f}], {sv["zoom"]:.1f}),'
        )
    maps_block = 'MAPS = [\n' + '\n'.join(lines) + '\n]'

    script_path = REPO / 'scripts' / 'export_maps.py'
    src = script_path.read_text(encoding='utf-8')
    new_src = re.sub(r'MAPS = \[.*?\]', maps_block, src, flags=re.DOTALL)
    script_path.write_text(new_src, encoding='utf-8')

    summary = ' | '.join(f'{pn}: z={sv["zoom"]:.1f}' for pn, sv in state.items())
    lbl_status.value = f'Guardado. {summary}'


dd_map.observe(load_map, names='value')
btn_prev.on_click(on_preview)
btn_save.on_click(on_save)

load_map()

display(w.VBox([
    dd_map,
    sl_zoom, sl_lat, sl_lon,
    w.HBox([btn_prev, btn_save]),
    lbl_status,
    img_widget,
]))